In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import optuna
import catboost
os.chdir('C:\\Users\\ferna\\OneDrive\\Downloads\\Code\\Python\\ML_Factor')

In [3]:
from data.data_load import *

## CatBoost - Tutorial

In [4]:
from catboost.datasets import amazon

In [6]:
(train_df, test_df) = amazon()

In [7]:
train_df.head()

,ACTION,RESOURCE,MGR_ID,ROLE_ROLLUP_1,ROLE_ROLLUP_2,ROLE_DEPTNAME,ROLE_TITLE,ROLE_FAMILY_DESC,ROLE_FAMILY,ROLE_CODE
0,1,39353,85475,117961,118300,123472,117905,117906,290919,117908
1,1,17183,1540,117961,118343,123125,118536,118536,308574,118539
2,1,36724,14457,118219,118220,117884,117879,267952,19721,117880
3,1,36135,5396,117961,118343,119993,118321,240983,290919,118322
4,1,42680,5905,117929,117930,119569,119323,123932,19793,119325


In [10]:
# train_df.describe() # 32769 obs
# test_df.describe()  # 58921 obs

In [13]:
y = train_df['ACTION']
X = train_df.drop('ACTION', axis = 1)

In [14]:
cat_features = list(range(0, X.shape[1]))
print(cat_features)

[0, 1, 2, 3, 4, 5, 6, 7, 8]


In [15]:
print('Labels: {}'.format(set(y)))
print('Zero count = {}, One count = {}'.format(len(y) - sum(y), sum(y)))

Labels: {0, 1}
Zero count = 1897, One count = 30872


In [18]:
from catboost import CatBoostClassifier
model = CatBoostClassifier(iterations=300)
model.fit(X,y,cat_features, verbose = 20)

Learning rate set to 0.137885
0:	learn: 0.5434076	total: 23.5ms	remaining: 7.04s
20:	learn: 0.1605809	total: 929ms	remaining: 12.3s
40:	learn: 0.1509985	total: 2.01s	remaining: 12.7s
60:	learn: 0.1478607	total: 3.14s	remaining: 12.3s
80:	learn: 0.1446394	total: 4.23s	remaining: 11.4s
100:	learn: 0.1424015	total: 5.35s	remaining: 10.5s
120:	learn: 0.1397514	total: 6.53s	remaining: 9.66s
140:	learn: 0.1377105	total: 7.73s	remaining: 8.71s
160:	learn: 0.1355154	total: 8.88s	remaining: 7.66s
180:	learn: 0.1338844	total: 9.99s	remaining: 6.57s
200:	learn: 0.1322251	total: 11.2s	remaining: 5.5s
220:	learn: 0.1304249	total: 12.3s	remaining: 4.4s
240:	learn: 0.1284430	total: 13.5s	remaining: 3.3s
260:	learn: 0.1267442	total: 14.7s	remaining: 2.19s
280:	learn: 0.1254264	total: 15.8s	remaining: 1.07s
299:	learn: 0.1237953	total: 16.9s	remaining: 0us


CatBoostClassifier(iterations=300)

In [19]:
model.predict_proba(X)

array([[0.00136477, 0.99863523],
       [0.00613328, 0.99386672],
       [0.02870183, 0.97129817],
       ...,
       [0.00665003, 0.99334997],
       [0.06138099, 0.93861901],
       [0.0200958 , 0.9799042 ]])

#### Pool class

* Instead of using X,y to pass dataset to training.
* Pool class is the class for storing a dataset.
* Obs: use Pool class if the dataset has more than just X and y (ex:sample weights or groups) or if the dataset is large.

In [20]:
from catboost import Pool
pool = Pool(data=X, label = y, cat_features = cat_features)

In [21]:
from sklearn.model_selection import train_test_split

data = train_test_split(X,y, test_size = 0.2, random_state = 0)
X_train, X_validation, y_train, y_validation = data

train_pool      = Pool(data = X_train, label = y_train, cat_features = cat_features)
validation_pool = Pool(data = X_validation, label = y_validation, cat_features = cat_features)

#### Loss function
* Binary $y \in {0,1}$ - use Logloss (caso particular da cross-entropy para classificação binária.)
* Probabilities $y \in [0,1]$- use CrossEntropy (caso geral, usado em multiclass)
* Both have the same formulation for binary responses:
$
  L = -\sum_{n=1}^{N}\left[y_n\log p_n + (1-y_n)\log(1-p_n)\right].
$

In [22]:
model = CatBoostClassifier(iterations = 15, learning_rate = 0.1,loss_function='Logloss')

In [23]:
model.fit(train_pool, eval_set=validation_pool, verbose=False)

CatBoostClassifier(iterations=15, learning_rate=0.1, loss_function='Logloss')

In [24]:
print('Model is fitted: {}'.format(model.is_fitted()))
print('Model params:\n{}'.format(model.get_params()))

Model is fitted: True
Model params:
{'iterations': 15, 'learning_rate': 0.1, 'loss_function': 'Logloss'}
